# RAG pipeline approach - Create a vector database
## Forewords
This first part will show you how to build a vector database, i.e., the “augmented” knowledge aimed to guide your LLM’s responses.

The simplified workflow will be as follows:

0. Create a proper working environment with all needed connections (s3: object storage, qdrant: vector database and llm.lab: LLM provider)
1. Take the NACE2.1 code definitions (code + title, + inclusions + exclusions in plain text)
2. Concatenate these pieces of information for each unique NACE (one code => 1 text)
3. Embed all these texts (1 text => 1 vector point)
4. Upload the points to Qdrant (the vector database service)


## Prepare your environment

### Exercise 1: Connections

#### Question 0
Load your secrets stored in the .env file into your environment.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

### Question 1
Create your llm.lab connection using openai client. Please name this client client_llmlab.
You will need to use your creds os.environ["LLMLAB_URL"] and os.environ["LLMLAB_API_KEY"] (see the previous chapter).

More information here (https://github.com/openai/openai-python) about OpenAI API.

(Go to next question to see the answer).

In [2]:
pip install openai

Note: you may need to restart the kernel to use updated packages.


### Question 2
Print all the available models (kindly made available to everyone by the SSPcloud team ❤️)
You can notice the llm.lab platform provides both generation and embedding models.

Tip
The OpenAI API provides a standard way to interact with language models for text generation and embeddings. Here, instead of OpenAI’s cloud, we connect to a remote OpenWebUI/vLLM server (the famous llm.lab from the even more famous SSPCloud), which hosts the models. The API interface stays the same, so you can generate text or create embeddings with calls like .completions.create() or .embeddings.create() while all computation happens on the OpenWebUI/vLLM server.

In [3]:
from openai import OpenAI

client_llmlab = OpenAI(
    base_url=os.environ["LLMLAB_URL"],
    api_key=os.environ["LLMLAB_API_KEY"],
)

# Print models list
models = client_llmlab.models.list()
for model in models.data:
    print(f"ID: {model.id}")

ID: gemma4-26b-moe
ID: qwen3-6-35b-moe
ID: qwen3-embedding-8b


### Question 3
- Create a connection to your Qdrant server. Please name it client_qdrant.
- Check all your existing collections (= databases). Probably not a single one for now.
Find the documentation here (https://qdrant.tech/documentation/) to dive deeper 😉.

In [4]:
pip install qdrant_client

Note: you may need to restart the kernel to use updated packages.


In [5]:
from qdrant_client import QdrantClient

# creating a connection
client_qdrant = QdrantClient(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"],
    port=os.environ["QDRANT_API_PORT"],
    check_compatibility=False
)

# checking collections available 
collections = client_qdrant.get_collections()
for collection in collections.collections:
    print(collection.name)

## Get and process NACE data
Time to import our data: the official NACE2.1 definitions!

The data are stored in the s3 storage system of the SSPCloud platform (uploaded from Eurostat (https://showvoc.op.europa.eu/#/datasets/ESTAT_Statistical_Classification_of_Economic_Activities_in_the_European_Community_Rev._2.1._%28NACE_2.1%29/downloads) )

## Exercise 2: handling NACE data
### Question 1
Import NACE data (for example, as a list of dictionaries with the structure below)

In [ ]:
import duckdb
con = duckdb.connect(database=":memory:")

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

path_nace = 'https://minio.lab.sspcloud.fr/projet-formation/diffusion/funathon/2026/project2/NACE_Rev2.1_Structure_Explanatory_Notes_EN.tsv'
query_definition = f"SELECT * FROM read_csv('{path_nace}')"
table = con.execute(query_definition).to_arrow_table()
nace = table.to_pylist()

nace[22]

{'ORDER_KEY': 230,
 'ID': '014',
 'CODE': '01.4',
 'HEADING': 'Animal production',
 'PARENT_ID': '01',
 'PARENT_CODE': '01',
 'LEVEL': 3,
 'Implementation_rule': None,
 'Includes': 'This group includes:\\n- farming (husbandry, raising) and breeding of all animals, except aquatic animals',
 'IncludesAlso': None,
 'Excludes': 'This group excludes:\\n- farm animal boarding and care, see 01.62\\n- production of hides and skins from slaughterhouses, see 10.11'}

### Question 2
- Build a class NaceDocument that will help you handle your NACE documents :

    - Cleans features (‘HEADING’, ‘Includes’, etc.) with a _clean method;
    - Checks your data (example: hierarchical coherence);
    - Creates a structured text for each code:
        - example: attribute text, created by a to_embedding_text method;
        - this text will be ready to be embedded;
        - It would be best to parametrize whether to include or not each possible field (e.g : text with or without “Excludes” feature).
Not familiar with @dataclass or @classmethod? See Python refresher.


Note 
Hint: structure of the `NaceDocument` class

```
# Structure of NaceDocument class
@dataclass
class NaceDocument:
    code: str
    heading: str
    level: int
    parent_code: Optional[str] = None
    includes: Optional[str] = None
    includes_also: Optional[str] = None
    excludes: Optional[str] = None

    text: str = field(init=False)

    @classmethod
    def from_raw(cls, raw: dict, with_includes_also=True, with_excludes=False,) -> "NaceDocument":
        # Write your method here

    def _clean(value) -> Optional[str]:
        # Write your method here

    def to_embedding_text(self, *, with_includes_also: bool = False, with_excludes: bool = False,) -> str:
        # Write your method here

# Example to instance a NaceDocument class

NaceDocument.from_raw(raw=nace_code,with_includes_also=True, with_excludes=True)
```

- Create all `NaceDocument` instances from `nace` list.
See an example of a cleaned and ready text for embedding:

```
# Code: 03.11
# Title: Marine fishing

## Includes:
This class includes:
- fishing on a commercial basis in ocean and coastal waters
- taking of marine crustaceans and molluscs
- whaling
- taking of marine aquatic animals (e.g. turtles, sea squirts, tunicates, sea urchins)

## Also includes:
This class also includes:
- gathering of other marine organisms and materials (e.g. natural pearls, sponges, coral, seaweed, algae)

## Excludes
This class excludes:
- frog farming, see 03.22\n
- operation of sport fishing preserves, see 93.19
```

In [10]:
from dataclasses import dataclass, field
from typing import Optional

def _clean(value) -> Optional[str]:
    """Normalize to stripped single-line string, or None if empty/missing."""
    if value is None:
        return None
    # str() handles non-string values (int, float...) from raw dicts
    # replace("\n", " ") flattens multiline strings to a single line
    # split() tokenizes on any whitespace, join(" ") rebuilds with single spaces
    cleaned = " ".join(str(value).replace("\n", " ").split())
    # Empty string is falsy in Python — return None instead for consistency
    return cleaned or None

@dataclass
class NaceDocument:
    code: str
    heading: str
    level: int
    parent_code: Optional[str] = None
    includes: Optional[str] = None
    includes_also: Optional[str] = None
    excludes: Optional[str] = None

    text: str = field(init=False)

    @classmethod
    def from_raw(cls, raw: dict, with_includes_also=True, with_excludes=False,) -> "NaceDocument":
        for key in ("CODE", "HEADING", "LEVEL"):
            if not raw.get(key):
                raise ValueError(f"Missing required field: {key}")

        level = int(raw["LEVEL"])
        if not (1 <= level <= 4):
            raise ValueError(f"Invalid level: {level}")

        obj = cls(
            code=str(raw["CODE"]).strip(),
            heading=_clean(raw["HEADING"]),
            level=level,
            parent_code=_clean(raw.get("PARENT_CODE")),
            includes=_clean(raw.get("Includes")),
            includes_also=_clean(raw.get("IncludesAlso")),
            excludes=_clean(raw.get("Excludes")),
        )

        obj.text = obj.to_embedding_text(
            with_includes_also=with_includes_also,
            with_excludes=with_excludes,
        )

        return obj

    def to_embedding_text(
        self,
        *,
        with_includes_also: bool = False,
        with_excludes: bool = False,
    ) -> str:
        parts = []

        parts.append(f"# Code: {self.code}")
        parts.append(f"# Title: {self.heading}")

        if self.includes:
            parts.append("")
            parts.append("## Includes:")
            parts.append(self.includes.strip())

        if with_includes_also and self.includes_also:
            parts.append("")
            parts.append("## Also includes:")
            parts.append(self.includes_also.strip())

        if with_excludes and self.excludes:
            parts.append("")
            parts.append("## Excludes:")
            parts.append(self.excludes.strip())

        output = "\n".join(parts)
        output = output.replace("\\n", "\n")

        return output.strip()

nace_documents = []
for nace_code in nace:
    nace_documents.append(
        NaceDocument.from_raw(
            raw=nace_code,
            with_includes_also=True,
            with_excludes=True
        )
    )

### Question 3
Create a text from a single Nace code example, with or without `Excludes` field

In [11]:
from pprint import pprint

i = 50
print(f"Printing index {i}:")

print("=============================================")
print("=============================================")

nace_example = nace[i]
doc_example = NaceDocument.from_raw(nace_example)

print("\nPrinting text to embed (WITH exclusions):")
_ = doc_example.to_embedding_text(
    with_includes_also=True,
    with_excludes=True,
)
print(doc_example.text)

print("=============================================")
print("=============================================")

print("\nPrinting text to embed (WITHOUT exclusions):")
_ = doc_example.to_embedding_text(
    with_includes_also=True,
    with_excludes=False,
)
print(doc_example.text)

Printing index 50:

Printing text to embed (WITH exclusions):
# Code: 03.11
# Title: Marine fishing

## Includes:
This class includes:
- fishing on a commercial basis in ocean and coastal waters
- taking of marine crustaceans and molluscs
- whaling
- taking of marine aquatic animals (e.g. turtles, sea squirts, tunicates, sea urchins)

## Also includes:
This class also includes:
- gathering of other marine organisms and materials (e.g. natural pearls, sponges, coral, seaweed, algae)

Printing text to embed (WITHOUT exclusions):
# Code: 03.11
# Title: Marine fishing

## Includes:
This class includes:
- fishing on a commercial basis in ocean and coastal waters
- taking of marine crustaceans and molluscs
- whaling
- taking of marine aquatic animals (e.g. turtles, sea squirts, tunicates, sea urchins)

## Also includes:
This class also includes:
- gathering of other marine organisms and materials (e.g. natural pearls, sponges, coral, seaweed, algae)


# Embed your NACE text descriptions
Now let’s choose an embedding model. We will use `qwen3-embedding-8b`, available on HuggingFace (like all other models provided by llm.lab).

For later use, we need to know the dimension of the vectors output by our model: 4096 (see the model card (https://huggingface.co/Qwen/Qwen3-Embedding-8B) on HuggingFace).

In [12]:
EMB_MODEL_NAME = "qwen3-embedding-8b"
emb_dim = 4096

## Exercise 3: create a Qdrant vector store and feed it with embedded NACE text descriptions

## PointStruct
`PointStruct` is a Pydantic model provided by the qdrant_client library that represents a single vector database entry. It bundles together the three components Qdrant needs to store and query a point:

- a unique id,
- a vector (the embedding),
- and a payload (arbitrary metadata as a dict).

Passing raw dicts directly to the Qdrant API would require manual serialization and validation. PointStruct handles that for you and ensures the data is correctly typed before being sent to the server. It’s essentially the contract between your Python code and the Qdrant collection.

## Important: Limitations of the teaching approach
Please note that this workflow is not optimized at all.

Here, we intentionally adopt a step-by-step approach to support learning. In a production-ready setup, however, the workflow would be batched and asynchronous, with embeddings fed into the Qdrant store as soon as they are generated.

Advanced users are welcome to give it a try.

Below, please find a non-exhaustive list of good practices for production-grade workflows :

- Batching: Process multiple documents per request to reduce overhead and improve throughput.
- Concurrency Limits: Restrict the number of simultaneous requests to avoid overwhelming the server.
- Retries: Implement automatic retries for transient network failures or server timeouts.
- Progressive Saves: Save embeddings or processed points incrementally to avoid losing work in case of interruption.
- Logging: Record progress, errors, and metadata for monitoring and debugging.
- Monitoring & Alerts: Track performance metrics, request latency, and error rates.
- Validation & Sanity Checks: Ensure embeddings are generated correctly and vectors are valid before inserting into your vector store.
- Resource Management: Release memory and close connections appropriately to avoid leaks in long-running pipelines.
- Idempotency: Use deterministic IDs for points (e.g., uuid5) so retries or reprocessing don’t create duplicates.
- Security & Secrets: Keep API keys and sensitive data secure and never hardcode them in your code.

# Tip 1: Python refresher: @dataclass and @classmethod
`@dataclass`: classes that hold data
When you build a class whose main purpose is to store data, writing the `__init__`, `__repr__` and `__eq__` methods by hand quickly becomes repetitive. The @dataclass decorator (introduced in Python 3.7) generates them automatically from your type annotations.

Without @dataclass — a lot of boilerplate just to store three fields:

````
class NaceDocument:
    def __init__(self, code, title, includes=None):
        self.code = code
        self.title = title
        self.includes = includes

    def __repr__(self):
        return f"NaceDocument(code={self.code!r}, title={self.title!r})"

    def __eq__(self, other):
        return (self.code, self.title, self.includes) == (other.code, other.title, other.includes)
```

With @dataclass — the same behavior, declared once:

```
from dataclasses import dataclass
from typing import Optional

@dataclass
class NaceDocument:
    code: str                            # required field
    title: str                           # required field
    includes: Optional[str] = None       # optional field, with default
```

A few things to know:

- The type annotations (str, Optional[str], …) are mandatory — @dataclass uses them to discover the fields.
- Fields with a default value must come after fields without one (same rule as regular function arguments).
- You can still add your own methods to a dataclass (e.g., to_embedding_text in NaceDocument).
- Useful variants: @dataclass(frozen=True) makes instances immutable (like a NamedTuple), @dataclass(order=True) adds comparison operators (<, >, …).

`@classmethod`: methods that act on the class, not on an instance
Most methods you write receive self — the instance the method is called on. A method decorated with @classmethod receives cls instead — the class itself.

```
class NaceDocument:
    def to_embedding_text(self) -> str:    # instance method: needs an existing object
        return f"{self.code} — {self.title}"

    @classmethod
    def from_raw(cls, raw: dict) -> "NaceDocument":   # class method: builds a new object
        return cls(code=raw["CODE"], title=raw["HEADING"])
```

The typical use case is to define alternative constructors — different ways of building an instance from different sources:

```
doc1 = NaceDocument(code="A01", title="Crop production")     # via __init__
doc2 = NaceDocument.from_raw({"CODE": "A01", "HEADING": "Crop production"})  # via classmethod
doc3 = NaceDocument.from_json(json_string)                   # another classmethod, if you wrote one
```

## Putting it together
In the NaceDocument class above, both decorators play complementary roles:

@dataclass removes the boilerplate of declaring fields and writing __init__.
@classmethod provides from_raw, a clean entry point that parses a raw dictionary (with validation, cleaning, etc.) and hands back a fully built NaceDocument.
This is a very common pattern in data pipelines: a lightweight container for the data + one or more classmethods to ingest it from external formats (CSV row, JSON, database record…).

Why cls instead of writing NaceDocument(...) directly? Using cls makes the method inheritance-aware: if SpecialNaceDocument inherits from NaceDocument, calling SpecialNaceDocument.from_raw(...) will correctly build a SpecialNaceDocument (not a NaceDocument), because cls resolves to the actual class on which the method was called.

```

```